# An LSTM backward pass in NumPy

An LSTM cell has four gates' worth of weights, two states and a product at every turn, so its
backward pass is where hand-written gradients most often go wrong. You will write one step of
the cell, its backward pass, and backpropagation through time over a whole sequence, and check
every gradient against central differences.

**How to work through it.** Each part asks for one function, or a small pair of
them in one cell. Replace each `TODO` and `raise NotImplementedError` with your
code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. The cell uses the lesson's equations. With $\mathbf{v}_t = [\mathbf{h}_{t-1}, \mathbf{x}_t]$,
the previous state and the input joined into one vector,

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(W_f \mathbf{v}_t + \mathbf{b}_f), &
\mathbf{i}_t &= \sigma(W_i \mathbf{v}_t + \mathbf{b}_i), &
\mathbf{o}_t &= \sigma(W_o \mathbf{v}_t + \mathbf{b}_o), &
\tilde{\mathbf{c}}_t &= \tanh(W_c \mathbf{v}_t + \mathbf{b}_c), \\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t, &
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t) .
\end{aligned}
$$

The four weight matrices are stacked into one `W` of shape `(4H, H + D)`, rows `0:H` for the
forget gate, `H:2H` for the input gate, `2H:3H` for the output gate and `3H:4H` for the
candidate, and the four biases into one `b` of length `4H`. Batches have one sequence per row.

In [ ]:
import numpy as np


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def numeric_grad(f, A, h=1e-6):
    """Central differences of f() in every entry of the array A. A is nudged in place and restored."""
    G = np.zeros_like(A)
    for idx in np.ndindex(A.shape):
        old = A[idx]
        A[idx] = old + h
        up = f()
        A[idx] = old - h
        down = f()
        A[idx] = old
        G[idx] = (up - down) / (2 * h)
    return G


def first_mismatch(pairs, atol=1e-7):
    """Name of the first (name, analytic, numeric) triple that disagrees, or None."""
    for name, got, num in pairs:
        if np.shape(got) != np.shape(num) or not np.allclose(got, num, atol=atol):
            return name
    return None


B, D, H = 2, 2, 3
gen = np.random.default_rng(8)
W = 0.6 * gen.normal(size=(4 * H, H + D))
b = 0.2 * gen.normal(size=4 * H)
print("W", W.shape, "b", b.shape)

## Part 1 (coding task): One step of the cell

Write `lstm_step(x, h_prev, c_prev, W, b)` for `x` of shape `(B, D)` and states of shape
`(B, H)`. It returns `(h, c, cache)`, the new hidden state, the new cell state, and a cache
holding whatever your backward pass in Part 2 will need.

In [ ]:
def lstm_step(x, h_prev, c_prev, W, b):
    """One LSTM step for a batch. Returns (h, c, cache)."""
    # TODO
    raise NotImplementedError

In [ ]:
Hc, Dc = 3, 2
Wc = 0.5 * np.sin(np.arange(4 * Hc * (Hc + Dc)).reshape(4 * Hc, Hc + Dc) * 0.37 + 0.2)
bc = 0.3 * np.cos(np.arange(4 * Hc) * 0.9)
xc = np.array([[0.5, -1.0], [1.5, 0.25]])
hc = np.array([[0.1, -0.3, 0.2], [0.0, 0.4, -0.1]])
cc = np.array([[0.5, -0.2, 1.0], [-0.7, 0.3, 0.0]])

h1, c1, _ = lstm_step(xc, hc, cc, Wc, bc)
want_h = np.array([[0.12791108, -0.16722611, 0.23991371], [-0.35847981, 0.32058302, 0.01542280]])
want_c = np.array([[0.25307779, -0.32838479, 0.42203880], [-0.92497180, 0.43645055, 0.03300309]])
assert np.shape(h1) == (2, 3) and np.shape(c1) == (2, 3), f"h and c should have shape (B, H) = (2, 3), got {np.shape(h1)} and {np.shape(c1)}"
assert np.allclose(c1, want_c, atol=1e-7), ("c is wrong: check that [h_prev, x] is joined in that order and the four blocks of W "
                                           "are used as forget, input, output, candidate")
assert np.allclose(h1, want_h, atol=1e-7), "c is right and h is not: h is the output gate times tanh(c)"
print("Part 1 passed")

## Part 2 (coding task): The backward pass of one step

Write `lstm_step_backward(dh, dc, cache)`. Its inputs are the loss's derivatives at this step's
outputs: `dh` is $\partial L / \partial \mathbf{h}_t$, and `dc` is the part of
$\partial L / \partial \mathbf{c}_t$ that comes through the later steps only, so it does not yet
include what reaches $\mathbf{c}_t$ through $\mathbf{h}_t$. It returns `(dx, dh_prev, dc_prev, dW, db)`, the gradients
with respect to `x`, `h_prev`, `c_prev`, `W` and `b`, each shaped like the array it belongs to.

In [ ]:
def lstm_step_backward(dh, dc, cache):
    """Gradients of one LSTM step: (dx, dh_prev, dc_prev, dW, db)."""
    # TODO
    raise NotImplementedError

In [ ]:
x1, hp, cp = gen.normal(size=(B, D)), gen.normal(size=(B, H)), gen.normal(size=(B, H))
Rh, Rc = gen.normal(size=(B, H)), gen.normal(size=(B, H))


def check_step(Rh, Rc):
    def L():
        h, c, _ = lstm_step(x1, hp, cp, W, b)
        return np.sum(h * Rh) + np.sum(c * Rc)
    _, _, cache = lstm_step(x1, hp, cp, W, b)
    dx, dhp, dcp, dW, db = lstm_step_backward(Rh, Rc, cache)
    return first_mismatch([("dc_prev", dcp, numeric_grad(L, cp)), ("dh_prev", dhp, numeric_grad(L, hp)),
                           ("dx", dx, numeric_grad(L, x1)), ("dW", dW, numeric_grad(L, W)),
                           ("db", db, numeric_grad(L, b))])


bad = check_step(np.zeros((B, H)), Rc)
assert bad is None, (f"with dh = 0, {bad} does not match the numeric gradient. The gradient enters only through "
                     "c_t = f * c_prev + i * candidate, so the output gate gets none: check each gate's error")
bad = check_step(Rh, np.zeros((B, H)))
assert bad is None, (f"with dc = 0, {bad} does not match the numeric gradient. The gradient enters through h_t only: "
                     "it reaches the output gate, and it reaches c_t through the slope of tanh(c_t)")
bad = check_step(Rh, Rc)
assert bad is None, f"with both dh and dc, {bad} does not match the numeric gradient"
print("Part 2 passed")

## Part 3 (coding task): Backpropagation through a sequence

Write two functions.

- `lstm_forward(X, h0, c0, W, b)` runs `lstm_step` over `X` of shape `(B, T, D)` and returns
  `(hs, caches)`: the hidden states of every step, shape `(B, T, H)`, and the list of the `T`
  step caches.
- `lstm_backward(dhs, caches)` takes `dhs` of shape `(B, T, H)`: at each step, the part of
  $\partial L / \partial \mathbf{h}_t$ that comes from outside the recurrence, for example from
  an output at every step. It returns `(dX, dh0, dc0, dW, db)`, the gradients with respect to `X`, `h0`,
  `c0`, `W` and `b`.

In [ ]:
def lstm_forward(X, h0, c0, W, b):
    """Run the cell over a sequence. Returns (hs of shape (B, T, H), list of step caches)."""
    # TODO
    raise NotImplementedError


def lstm_backward(dhs, caches):
    """Backpropagation through time. Returns (dX, dh0, dc0, dW, db).

    Run the loop over every step, T - 1 down to 0, so step 0 is in it as well:
    dW and db add up one contribution per step. Each step fills its own slot of
    dX with that step's dx of shape (B, D), and all T slots are filled before
    they stack into dX of shape (B, T, D).
    """
    # TODO
    raise NotImplementedError

In [ ]:
T = 5
X = gen.normal(size=(B, T, D))
h0, c0 = gen.normal(size=(B, H)), gen.normal(size=(B, H))


def shape_check(name, got, want, note=""):
    try:
        shape = np.shape(got)
    except (IndexError, TypeError, ValueError) as exc:
        why = f"np.shape raised {type(exc).__name__}: {exc}".rstrip(".")
        raise AssertionError(f"{name} cannot be read as one array of shape {want}: {why}{note}") from None
    assert shape == want, f"{name} has shape {shape}, expected {want}{note}"


def check_sequence(R):
    def L():
        return np.sum(lstm_forward(X, h0, c0, W, b)[0] * R)
    hs, caches = lstm_forward(X, h0, c0, W, b)
    assert np.shape(hs) == (B, T, H) and len(caches) == T, f"expected hs of shape {(B, T, H)} and {T} caches"
    try:
        dX, dh0, dc0, dW, db = lstm_backward(R, caches)
    except (IndexError, TypeError, ValueError) as exc:
        why = f"lstm_backward(R, caches) raised {type(exc).__name__}: {exc}".rstrip(".")
        raise AssertionError(f"{why}. dx is collected step by step, {T} slots of shape {(B, D)} with one per "
                             "step, and every slot is filled before they stack into dX") from None
    shape_check("dX", dX, (B, T, D), f". dX is collected step by step, {T} pieces of shape {(B, D)} with one "
                f"per step, and all of them are there before they stack into dX of shape {(B, T, D)}")
    shape_check("dh0", dh0, (B, H))
    shape_check("dc0", dc0, (B, H))
    shape_check("dW", dW, (4 * H, H + D))
    shape_check("db", db, (4 * H,))
    num_dW, num_db, num_dX = numeric_grad(L, W), numeric_grad(L, b), numeric_grad(L, X)
    wrong = [t for t in range(T) if not np.allclose(dX[:, t], num_dX[:, t], atol=1e-7)]
    assert not wrong, (f"dX is wrong at step(s) {wrong}: the loop has to run over every step from 0 to {T - 1}, and "
                       "each step has to write its own dx into its own slot, so that no step is skipped and no slot "
                       "is left unfilled")
    bad = first_mismatch([("dW", dW, num_dW), ("db", db, num_db), ("dX", dX, num_dX),
                          ("dh0", dh0, numeric_grad(L, h0)), ("dc0", dc0, numeric_grad(L, c0))])
    if bad in ("dW", "db"):
        return (f"{bad} ({bad} adds up one contribution per step, so the loop has to cover every step from 0 to "
                f"{T - 1}, not 1 to {T - 1}, and add that step's own contribution)")
    return bad


# A loss on the last step only, then a loss on every step.
R_last = np.zeros((B, T, H))
R_last[:, -1] = gen.normal(size=(B, H))
bad = check_sequence(R_last)
assert bad is None, (f"with a loss on the last step only, {bad} does not match the numeric gradient: "
                     "check what each step passes back to the one before, both dh and dc, and that dW and db add up over steps")
bad = check_sequence(gen.normal(size=(B, T, H)))
assert bad is None, (f"with a loss on every step, {bad} does not match the numeric gradient: "
                     "h_t is read by the loss at step t and by step t + 1, so its gradient is the sum of both")
print("Part 3 passed")

## Solution

### Part 1: One step of the cell

In [ ]:
def lstm_step(x, h_prev, c_prev, W, b):
    """One LSTM step for a batch. Returns (h, c, cache)."""
    H = h_prev.shape[1]
    v = np.concatenate([h_prev, x], axis=1)            # (B, H + D)
    a = v @ W.T + b                                     # (B, 4H): the four pre-activations
    f = sigmoid(a[:, :H])
    i = sigmoid(a[:, H:2 * H])
    o = sigmoid(a[:, 2 * H:3 * H])
    g = np.tanh(a[:, 3 * H:])                           # the candidate
    c = f * c_prev + i * g
    h = o * np.tanh(c)
    return h, c, (v, c_prev, f, i, o, g, c, W)

**Why it works.** One matrix product computes all four pre-activations, and slicing it
into blocks of `H` columns gives each gate its own. The joined vector must put $\mathbf{h}_{t-1}$ first,
because the columns of `W` were laid out that way, and the candidate is the only block that uses
tanh. The cache keeps the joined input, the old cell, the four gate values and the new cell:
every derivative in the backward pass is a product of these.

### Part 2: The backward pass of one step

In [ ]:
def lstm_step_backward(dh, dc, cache):
    """Gradients of one LSTM step: (dx, dh_prev, dc_prev, dW, db)."""
    v, c_prev, f, i, o, g, c, W = cache
    H = c.shape[1]
    tc = np.tanh(c)
    dc = dc + dh * o * (1.0 - tc ** 2)                 # all that reaches c_t, through h_t as well
    da = np.concatenate([
        dc * c_prev * f * (1.0 - f),                    # forget gate, through its sigmoid
        dc * g * i * (1.0 - i),                         # input gate
        dh * tc * o * (1.0 - o),                        # output gate
        dc * i * (1.0 - g ** 2),                        # candidate, through its tanh
    ], axis=1)
    dv = da @ W
    return dv[:, H:], dv[:, :H], dc * f, da.T @ v, da.sum(axis=0)

**Why it works.** The gradient reaching $\mathbf{c}_t$ comes from two places: `dc` from the
next step's cell path, and `dh` through $\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$, which
adds $\mathbf{o}_t \odot (1 - \tanh^2\mathbf{c}_t) \odot$ `dh`. Their sum is the error of the cell. From there every
gate is one product away: the forget gate multiplied $\mathbf{c}_{t-1}$, the input gate multiplied the
candidate, and the output gate multiplied $\tanh(\mathbf{c}_t)$. Each then passes through the slope of its
own squashing function, $s(1 - s)$ for a sigmoid with value $s$ and $1 - g^2$ for the tanh
candidate with value $g$. The four errors stacked are the gradient of the four pre-activations,
so `dW` is one outer product with the joined input and `dv` splits back into `dh_prev` and `dx`.
`dc_prev` is the cell's error times the forget gate, the cell path from the lesson. Testing with
`dh = 0` and then `dc = 0` separates the two routes, so a slip in one is not hidden by the other.

### Part 3: Backpropagation through a sequence

In [ ]:
def lstm_forward(X, h0, c0, W, b):
    """Run the cell over a sequence. Returns (hs of shape (B, T, H), list of step caches)."""
    h, c = h0, c0
    hs, caches = [], []
    for t in range(X.shape[1]):
        h, c, cache = lstm_step(X[:, t], h, c, W, b)
        hs.append(h)
        caches.append(cache)
    return np.stack(hs, axis=1), caches


def lstm_backward(dhs, caches):
    """Backpropagation through time. Returns (dX, dh0, dc0, dW, db).

    Run the loop over every step, T - 1 down to 0, so step 0 is in it as well:
    dW and db add up one contribution per step. Each step fills its own slot of
    dX with that step's dx of shape (B, D), and all T slots are filled before
    they stack into dX of shape (B, T, D).
    """
    Bn, T, Hn = dhs.shape
    dh_next, dc_next = np.zeros((Bn, Hn)), np.zeros((Bn, Hn))
    dX, dW, db = [None] * T, 0.0, 0.0
    for t in reversed(range(T)):
        # h_t is read by the outside loss and by step t + 1, so its gradient is the sum.
        dx, dh_next, dc_next, dW_t, db_t = lstm_step_backward(dhs[:, t] + dh_next, dc_next, caches[t])
        dX[t] = dx
        dW, db = dW + dW_t, db + db_t
    return np.stack(dX, axis=1), dh_next, dc_next, dW, db

**Why it works.** The backward loop runs the steps in reverse and carries two gradients
from step $t + 1$ to step $t$: `dh_next` and `dc_next`. The state $\mathbf{h}_t$ is read both by whatever
sits on top of the layer at step $t$ and by step $t + 1$, so the step's backward pass receives the
sum, `dhs[:, t] + dh_next`. The cell state is read only by step $t + 1$, through its cell path and
through $\mathbf{h}_t$, which is why `dc_next` goes in on its own. `W` and `b` are used at every step, so
their gradients add up over the steps. A loss on the last step alone tests the carried gradients
by themselves, since every other step's `dhs` is zero, and a loss on every step then tests the sum.